# Train the climbing-hold detector on Colab

Fine-tune `yolo11s-seg` on the prepared dataset, using a free Colab GPU. This
notebook is a thin driver: all training logic lives in `scripts/train.py`
(same code that runs on the laptop), so there is nothing here to keep in sync.

**Before you start**
- Runtime → Change runtime type → **GPU** (T4 is plenty).
- Prepare the dataset locally with `scripts/prepare_dataset.py`, then upload to a
  folder on your Drive (the `DRIVE_DIR` below):
  - `dataset.zip` — the zipped dataset (zip root holds `train/ val/ test/`)
  - `dataset.yaml` — the config emitted by the prep script

**Surviving Colab session limits:** runs land in `RUN_DIR` on Drive, so a killed
session loses nothing — re-run the setup cells, then the **Resume** cell (5b)
instead of the fresh-train cell (5a).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config

Everything Drive-side is anchored to `DRIVE_DIR`. Training writes runs straight
to `RUN_DIR` on Drive so checkpoints persist across sessions.

In [ ]:
import os

# --- edit these to match your Drive layout ---
REPO_URL  = 'https://github.com/antonylgs/yolov11-climbing-holds-detector.git'
DRIVE_DIR = '/content/drive/MyDrive/climbing-holds-detector'   # holds dataset.zip + dataset.yaml + runs/
EPOCHS    = 150
IMGSZ     = 1280
BATCH     = 16          # safe at imgsz 1280 on A100; bump to 24/32 if epoch 1 has VRAM headroom
PATIENCE = 30          # early-stop if val mAP doesn't improve for this many epochs

DATA_YAML = f'{DRIVE_DIR}/dataset.yaml'        # the config emitted by prepare_dataset.py
DATASET   = f'{DRIVE_DIR}/data/dataset'        # used only if you upload an unzipped folder
DATASET_ZIP = f'{DRIVE_DIR}/dataset.zip'       # preferred: one zip is far faster to upload
RUN_DIR   = f'{DRIVE_DIR}/runs'                # checkpoints + curves persist here
RUN_NAME  = 'holds'

assert os.path.exists(DATA_YAML), f'dataset.yaml not found on Drive: {DATA_YAML}'
assert os.path.exists(DATASET_ZIP) or os.path.isdir(DATASET), \
    f'upload dataset.zip (or a data/dataset/ folder) to {DRIVE_DIR}'
print('config OK')

## 3. Get the code + install deps

Clone the repo (or `git pull` if it's already there) and install it. We install
the package so `scripts/train.py` imports `holds_detector.train` cleanly.

In [ ]:
import os
if not os.path.isdir('/content/climbing-holds-detector'):
    !git clone {REPO_URL} /content/climbing-holds-detector
else:
    !cd /content/climbing-holds-detector && git pull --ff-only
%cd /content/climbing-holds-detector
!pip install -q -e .

## 4. Stage the dataset locally

Training off Drive directly is slow (every image read crosses the network). Unzip
(or copy) the dataset onto Colab's fast local disk, then rewrite `dataset.yaml`'s
`path:` to point at the local copy. Runs still go to Drive so they persist.

In [ ]:
import shutil, pathlib, zipfile

LOCAL_DATASET = '/content/dataset'
if not os.path.isdir(LOCAL_DATASET):
    if os.path.exists(DATASET_ZIP):
        print('unzipping dataset.zip from Drive…')
        with zipfile.ZipFile(DATASET_ZIP) as zf:
            zf.extractall(LOCAL_DATASET)   # zip root holds train/ val/ test/
    else:
        print('copying dataset folder from Drive…')
        shutil.copytree(DATASET, LOCAL_DATASET)
assert os.path.isdir(f'{LOCAL_DATASET}/train/images'), 'unexpected dataset layout'

LOCAL_YAML = '/content/dataset.yaml'
src = pathlib.Path(DATA_YAML).read_text().splitlines()
lines = [f'path: {LOCAL_DATASET}' if ln.startswith('path:') else ln for ln in src]
pathlib.Path(LOCAL_YAML).write_text('\n'.join(lines) + '\n')
print(pathlib.Path(LOCAL_YAML).read_text())

## 5a. Train (fresh run)

Calls the exact same entrypoint as the laptop. `--device` is auto-selected
(CUDA here). `--no-export` skips copying `best.pt` into the repo's `models/` — we
pull the weights off Drive at the end instead.

In [ ]:
!python scripts/train.py \
    --data {LOCAL_YAML} \
    --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} \
    --patience {PATIENCE} \
    --project "{RUN_DIR}" --name {RUN_NAME} \
    --no-export

## 5b. Resume (after a dropped session)

Run this **instead of 5a** if the session died mid-run. With no path, it resumes
the most recently modified `last.pt` under `RUN_DIR`; Ultralytics continues with
that run's stored args, so epochs/imgsz are inherited — you don't repeat them.

In [ ]:
!python scripts/train.py --resume --project "{RUN_DIR}" --no-export

## 6. Collect results

Runs already live on Drive (`RUN_DIR`), so `best.pt`, `results.png`, `args.yaml`,
and `run_meta.json` are persisted. This cell just locates the latest run and
prints the key paths to download / point inference at.

In [ ]:
import glob, os
runs = sorted(glob.glob(f'{RUN_DIR}/**/weights/best.pt', recursive=True), key=os.path.getmtime)
assert runs, f'no best.pt under {RUN_DIR}'
best = runs[-1]
run = os.path.dirname(os.path.dirname(best))
print('best weights :', best)
print('result curves:', os.path.join(run, 'results.png'))
print('run args     :', os.path.join(run, 'args.yaml'))
print('run meta     :', os.path.join(run, 'run_meta.json'))
print('\nDownload best.pt into the repo as models/best.pt to run the CLI locally.')